# Ivan DE Qwen baseline — LRZ web Jupyter (Open OnDemand)

Use this notebook at **https://login.ai.lrz.de** → **Interactive Apps** → **Jupyter Notebook**.

## Important: start a **GPU** session (not CPU only)

Your browser URL must contain a **GPU node** (e.g. `gpu-…`), **not** `cpu-003` like a CPU-only session.

1. Open **My Interactive Sessions** and **delete** any running CPU-only Jupyter session.
2. **Interactive Apps** → **Jupyter Notebook**
3. Set **Resources: CPU + single GPU** (not “CPU only”)
4. Pick a **PyTorch / CUDA** container, enough RAM (≥32 GB), runtime (e.g. 4 h)
5. **Launch** → when **Running**, click **Connect to Jupyter**

## Files in the same Jupyter folder

Upload next to this notebook (via **Upload** in the file browser):

| File | Required |
|------|----------|
| `ende_dev_v2.jsonl` | Yes |
| `Ivan_de_qwen_baseline.ipynb` | This file |

Paste your **`HF_TOKEN` in cell 2** (LRZ has no `scripts/.env` needed).

## Run order

| Cell | What |
|------|------|
| 1 | Install **pinned** packages (do not upgrade `torch`; then **Restart kernel**) |
| 2 | Load Qwen on GPU |
| 3 | Paths (`ende_dev_v2.jsonl`, `qwen_outputs/`) |
| 4–7 | Metrics, translate, evaluate |

Set `MAX_SAMPLES = 10` in cell 3 for a quick test.

Outputs: `qwen_outputs/` in this folder.

## If cell 2 fails with `device_mesh` / `GenerationMixin`

The LRZ container’s **torch** is older than the newest **transformers**. Cell 1 installs a compatible transformers (`<4.46`). After cell 1: **Kernel → Restart**, then run cells 2–7.


In [ ]:
# Cell 1: Install dependencies (run once per Jupyter session)
# LRZ containers ship an older torch — do NOT pip install torch.
# Latest transformers needs torch.distributed.tensor.device_mesh and will crash.

%pip install -q "transformers>=4.40.0,<4.46.0" "accelerate>=0.33.0,<1.0.0" sacrebleu tqdm huggingface_hub

import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

# Quick import test (fails early if versions are incompatible)
from transformers import AutoModelForCausalLM, AutoTokenizer
print("transformers import OK — now use Kernel → Restart, then run from cell 2")


In [ ]:
# ============================================================
# Cell 2: Setup — cache, env, load Qwen on GPU
# ============================================================

import os
import warnings
from pathlib import Path

# ===== PASTE YOUR HUGGING FACE TOKEN HERE (LRZ: no .env file) =====
HF_TOKEN = ""  # e.g. "hf_xxxxxxxx..."
# ===================================================================

os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

if HF_TOKEN and HF_TOKEN not in ("", "your_token_here", "hf_your-token-here"):
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("HF_TOKEN set from notebook")
else:
    print("HF_TOKEN not set (optional for public Qwen models)")


def is_lrz_environment() -> bool:
    if os.environ.get("SCRATCH") or os.environ.get("WORK"):
        return True
    if os.environ.get("SLURM_JOB_ID"):
        return True
    host = (os.environ.get("HOSTNAME") or "").lower()
    return "lrz" in host or "ai.lrz.de" in host


def is_ood_cpu_only_node() -> bool:
    """OOD CPU-only Jupyter sessions use hostnames like cpu-003.ai.lrz.de."""
    host = (os.environ.get("HOSTNAME") or "").lower()
    return "cpu-" in host and "ai.lrz.de" in host


def find_repo_root() -> Path:
    """Project root, or Jupyter working directory when only notebook + jsonl are uploaded."""
    env_root = os.environ.get("TERMINOLOGY_REPO", "").strip()
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if root.is_dir():
            return root
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "scripts").is_dir() or (path / "Baseline").is_dir():
            return path
    cwd = Path.cwd()
    if (cwd / "ende_dev_v2.jsonl").is_file():
        return cwd
    if (cwd / "Baseline" / "ende_dev_v2.jsonl").is_file():
        return cwd
    return cwd


def setup_huggingface_cache(repo_root: Path) -> Path:
    if os.environ.get("HF_HOME"):
        cache_root = Path(os.environ["HF_HOME"]).expanduser()
    elif os.environ.get("SCRATCH"):
        cache_root = Path(os.environ["SCRATCH"]) / "huggingface_cache"
    elif is_lrz_environment():
        cache_root = Path.home() / ".cache" / "huggingface"
    else:
        cache_root = repo_root / ".cache" / "huggingface"

    cache_root.mkdir(parents=True, exist_ok=True)
    hub_cache = cache_root / "hub"
    hub_cache.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(cache_root)
    os.environ["HUGGINGFACE_HUB_CACHE"] = str(hub_cache)
    return cache_root


REPO_ROOT = find_repo_root()
HF_CACHE = setup_huggingface_cache(REPO_ROOT)
print("Hostname:", os.environ.get("HOSTNAME"))
print("LRZ:", is_lrz_environment(), "| OOD CPU-only node:", is_ood_cpu_only_node())
print("Working directory:", Path.cwd())
print("Repo root:", REPO_ROOT)
print("HF cache:", HF_CACHE)

import torch

if torch.cuda.is_available():
    print("CUDA:", torch.cuda.get_device_name(0))
    print("CUDA memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
elif is_ood_cpu_only_node():
    raise RuntimeError(
        "This Jupyter session has NO GPU (hostname contains 'cpu-').\n"
        "Stop the session: OOD → My Interactive Sessions → Delete.\n"
        "Start again: Interactive Apps → Jupyter Notebook → Resources: CPU + single GPU.\n"
        "Then re-open this notebook and run from cell 1."
    )
elif is_lrz_environment():
    raise RuntimeError(
        "No CUDA GPU visible. For web Jupyter at LRZ, launch a GPU session (see markdown at top)."
    )
else:
    print("Warning: no GPU — CPU only (very slow).")

import transformers
print("transformers:", transformers.__version__)

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"{e}\n\nFix: run cell 1, then Kernel → Restart, then run cell 2 again. "
        "Do not install the latest transformers on LRZ (needs a newer torch)."
    ) from e

MODEL_NAME = os.environ.get("QWEN_MODEL", "Qwen/Qwen2.5-Coder-3B-Instruct")
LOCAL_MODEL_DIR = os.environ.get("QWEN_MODEL_DIR", "").strip()
model_source = LOCAL_MODEL_DIR if LOCAL_MODEL_DIR else MODEL_NAME

load_kwargs = {"torch_dtype": "auto", "cache_dir": str(HF_CACHE)}
load_kwargs["device_map"] = "auto" if torch.cuda.is_available() else "cpu"

print(f"Loading model from: {model_source}")
model = AutoModelForCausalLM.from_pretrained(model_source, **load_kwargs)
tokenizer = AutoTokenizer.from_pretrained(model_source, cache_dir=str(HF_CACHE))
print("Model ready.")


In [ ]:
# ============================================================
# Cell 3: Paths and settings
# ============================================================

# Optional: set absolute path if auto-search fails (LRZ file browser root is often "/")
DATA_FILE_OVERRIDE = ""  # e.g. "/ende_dev_v2.jsonl"

def _candidate_search_roots(repo_root: Path) -> list:
    """LRZ OOD: files often live in / but kernel cwd may be /workspace."""
    roots = [Path.cwd(), repo_root, Path("/"), Path("/workspace")]
    for key in ("HOME", "JUPYTER_SERVER_ROOT"):
        val = os.environ.get(key)
        if val:
            roots.append(Path(val))
    p = Path.cwd()
    for _ in range(6):
        roots.append(p)
        if p.parent == p:
            break
        p = p.parent
    seen = set()
    unique = []
    for r in roots:
        try:
            resolved = r.resolve()
        except OSError:
            continue
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def resolve_data_and_output(repo_root: Path):
    """Find ende_dev_v2.jsonl in cwd, repo, /workspace, or / (OOD layout)."""
    rel_paths = [
        Path("ende_dev_v2.jsonl"),
        Path("Baseline") / "ende_dev_v2.jsonl",
    ]
    tried = []
    for root in _candidate_search_roots(repo_root):
        for rel in rel_paths:
            data_file = root / rel
            tried.append(str(data_file))
            if data_file.is_file():
                output_dir = data_file.parent / "qwen_outputs"
                return data_file.resolve(), output_dir.resolve()
    msg = (
        "ende_dev_v2.jsonl not found.\n"
        f"  cwd = {Path.cwd()}\n"
        f"  REPO_ROOT = {repo_root}\n"
        "  Checked:\n    " + "\n    ".join(tried[:12])
    )
    if len(tried) > 12:
        msg += f"\n    ... and {len(tried) - 12} more paths"
    raise FileNotFoundError(msg)


if DATA_FILE_OVERRIDE:
    DATA_FILE = Path(DATA_FILE_OVERRIDE).expanduser().resolve()
    if not DATA_FILE.is_file():
        raise FileNotFoundError(f"DATA_FILE_OVERRIDE not found: {DATA_FILE}")
    OUTPUT_DIR = DATA_FILE.parent / "qwen_outputs"
else:
    DATA_FILE, OUTPUT_DIR = resolve_data_and_output(REPO_ROOT)

OUTPUT_DIR = Path(OUTPUT_DIR)

MAX_SAMPLES = None  # set to 10 for a quick OOD test
# MAX_SAMPLES = 10

MODES = ["no_term", "proper_term", "random_term"]
TARGET_LANG = "German"
OUTPUT_TAG = "deu"
REF_FIELD = "de"
TOP_TERM_PREVIEW = 5
TOP_SAMPLE_PREVIEW = 2
DATA_STEM = DATA_FILE.stem

print("DATA_FILE:", DATA_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# ============================================================
# Cell 3: Helper functions
# ============================================================

import json
import re
import sacrebleu

from typing import List, Dict, Optional
from collections import defaultdict, Counter
from tqdm import tqdm


def load_jsonl(path: str) -> List[Dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def save_jsonl(path: str, records: List[Dict]):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def strip_output_tags(text: str) -> str:
    """Remove translation output tags such as <deu>...</deu>."""
    if not isinstance(text, str):
        return text
    return re.sub(r"</?(deu|de|ger|german)>", "", text, flags=re.IGNORECASE).strip()


def compute_bleu_chrf(hyps: List[str], refs: List[str]) -> Dict:
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    return {
        "bleu": bleu.score,
        "chrf": chrf.score,
    }


def _normalize_text(text: str) -> str:
    return " ".join(str(text).lower().split())


def _count_term_occurrences(text: str, term: str) -> int:
    text_norm = _normalize_text(text)
    term_norm = _normalize_text(term)
    pattern = r"\b" + re.escape(term_norm) + r"\b"
    return len(re.findall(pattern, text_norm))


def terminology_accuracy_advanced(
    preds: List[str],
    samples: List[Dict],
    mode: str = "proper_term"
) -> Dict:
    term_ratios = {}
    total_terms = 0

    for pred, sample in zip(preds, samples):
        if mode == "proper_term":
            terms = (sample.get("proper_terms") or {}).copy()
        elif mode == "random_term":
            terms = (sample.get("random_terms") or {}).copy()
            for key in (sample.get("proper_terms") or {}).keys():
                terms.pop(key, None)
        else:
            terms = {}

        source_text = sample.get("en", "")

        for src, tgt in terms.items():
            total_terms += 1

            src_count = _count_term_occurrences(source_text, src)
            if src_count == 0:
                src_count = 1

            tgt_count = _count_term_occurrences(pred, tgt)
            ratio = min(tgt_count / src_count, 1.0)

            term_ratios[src] = ratio

    avg_accuracy = (
        sum(term_ratios.values()) / len(term_ratios) * 100
        if term_ratios
        else None
    )

    return {
        "total_terms": total_terms,
        "avg_ratio_pct": avg_accuracy,
        "per_term_ratios": term_ratios,
    }


def terminology_consistency_advanced(
    preds: List[str],
    samples: List[Dict],
    mode: str = "proper_term"
) -> Dict:
    term_to_candidates = defaultdict(list)

    for pred, sample in zip(preds, samples):
        if mode == "proper_term":
            terms = (sample.get("proper_terms") or {}).copy()
        elif mode == "random_term":
            terms = (sample.get("random_terms") or {}).copy()
            for key in (sample.get("proper_terms") or {}).keys():
                terms.pop(key, None)
        else:
            terms = {}

        for src, tgt in terms.items():
            if str(tgt).lower() in str(pred).lower():
                term_to_candidates[src].append(tgt)
            else:
                term_to_candidates[src].append("<MISSING>")

    pseudo_references = {}

    for src, candidates in term_to_candidates.items():
        counter = Counter(candidates)
        pseudo_references[src] = counter.most_common(1)[0][0]

    per_term_consistency = {}
    macro_scores = []
    weighted_scores = []

    for src, candidates in term_to_candidates.items():
        pseudo_ref = pseudo_references[src]
        matches = sum(1 for c in candidates if c == pseudo_ref)
        consistency = matches / len(candidates) if candidates else 0.0

        per_term_consistency[src] = {
            "occ": len(candidates),
            "pseudo_ref": pseudo_ref,
            "matches": matches,
            "consistency": consistency,
        }

        macro_scores.append(consistency)
        weighted_scores.extend([consistency] * len(candidates))

    return {
        "per_term": per_term_consistency,
        "macro_avg_consistency": (
            sum(macro_scores) / len(macro_scores)
            if macro_scores
            else None
        ),
        "weighted_avg_consistency": (
            sum(weighted_scores) / len(weighted_scores)
            if weighted_scores
            else None
        ),
    }


def _fmt_metric(value, digits=2):
    if value is None:
        return "N/A"
    return f"{value:.{digits}f}"


def _preview_term_stats(term_stats, limit=5):
    if not term_stats:
        return []

    items = sorted(
        term_stats.items(),
        key=lambda item: (-item[1].get("occ", 0), item[0])
    )

    return items[:limit]


In [ ]:
# ============================================================
# Cell 4: Local Qwen English → German translation function
# ============================================================

from typing import Optional


def translate_sample(
    sample_en: str,
    terminology: Optional[Dict[str, str]] = None,
    target_lang: str = TARGET_LANG,
    output_tag: str = OUTPUT_TAG,
    max_new_tokens: int = 256,
) -> str:
    term_block = ""

    if terminology:
        term_block = "Terminology:\n"
        for src, tgt in terminology.items():
            term_block += f"{src} -> {tgt}\n"
        term_block += "\n"

    prompt = f"""
You are a translation assistant.

Translate the English text to {target_lang}.

Rules:
1. Output only in this format: <{output_tag}> ... </{output_tag}>
2. Use the terminology mappings exactly as provided.
3. Do not explain anything.
4. Translate only from English to German.

{term_block}
Input:
<en> {sample_en} </en>
"""

    messages = [
        {
            "role": "system",
            "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant.",
        },
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
        )
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip()


In [ ]:
# ============================================================
# Cell 5: Load JSONL dataset
# ============================================================

samples = load_jsonl(str(DATA_FILE))

if MAX_SAMPLES is not None:
    samples = samples[:MAX_SAMPLES]

print(f"Loaded samples: {len(samples)}")
print("First sample keys:", samples[0].keys())

if "en" not in samples[0]:
    raise KeyError("Expected source field 'en' was not found.")

if REF_FIELD not in samples[0]:
    print(f"Warning: reference field '{REF_FIELD}' not found. BLEU/chrF cannot be computed.")
else:
    print(f"Reference field found: '{REF_FIELD}'")


In [ ]:
# ============================================================
# Cell 6: Run EN → DE translation and evaluation (sequential modes)
# ============================================================


def terminology_for_mode(sample: Dict, mode: str) -> Optional[Dict[str, str]]:
    if mode == "no_term":
        return None
    if mode == "proper_term":
        return sample.get("proper_terms") or None
    if mode == "random_term":
        terminology = (sample.get("random_terms") or {}).copy()
        for key in (sample.get("proper_terms") or {}).keys():
            terminology.pop(key, None)
        return terminology or None
    return None


def term_eval_mode(mode: str) -> str:
    if mode == "proper_term":
        return "proper_term"
    if mode == "random_term":
        return "random_term"
    return "no_term"


def run_mode_evaluation(
    mode: str,
    samples: List[Dict],
    base_output_dir: Path,
) -> Dict:
    print(f"\n--- Mode: {mode} (started) ---")

    preds = []
    records = []

    for sample in tqdm(samples, desc=f"{DATA_STEM} - {mode}"):
        pred = translate_sample(
            sample_en=sample.get("en", ""),
            terminology=terminology_for_mode(sample, mode),
            target_lang=TARGET_LANG,
            output_tag=OUTPUT_TAG,
        )
        preds.append(pred)

        record = sample.copy()
        record[f"prediction_{mode}"] = pred
        record[f"prediction_{mode}_clean"] = strip_output_tags(pred)
        records.append(record)

    clean_preds = [strip_output_tags(p) for p in preds]
    output_jsonl = base_output_dir / f"{DATA_STEM}_{mode}_predictions.jsonl"
    save_jsonl(str(output_jsonl), records)
    print(f"[{mode}] Saved predictions to: {output_jsonl}")

    metrics = {}

    if REF_FIELD in samples[0]:
        refs = [sample.get(REF_FIELD, "") for sample in samples]
        metrics.update(compute_bleu_chrf(clean_preds, refs))

        term_mode = term_eval_mode(mode)
        term_acc = terminology_accuracy_advanced(clean_preds, samples, mode=term_mode)
        term_cons = terminology_consistency_advanced(clean_preds, samples, mode=term_mode)

        metrics["terminology_accuracy"] = term_acc
        metrics["terminology_consistency"] = term_cons

        print(f"[{mode}] BLEU: {_fmt_metric(metrics['bleu'])}")
        print(f"[{mode}] chrF2++: {_fmt_metric(metrics['chrf'])}")
        print(
            f"[{mode}] Terminology accuracy ratio %: "
            f"{_fmt_metric(term_acc.get('avg_ratio_pct'))}"
        )
        print(f"[{mode}] Terminology terms counted: {term_acc.get('total_terms', 0)}")
        print(
            f"[{mode}] Macro-avg consistency: "
            f"{_fmt_metric(term_cons.get('macro_avg_consistency'))}"
        )
        print(
            f"[{mode}] Weighted-avg consistency: "
            f"{_fmt_metric(term_cons.get('weighted_avg_consistency'))}"
        )

        if term_acc.get("per_term_ratios"):
            print(f"[{mode}] Top terminology accuracy terms:")
            preview_data = {
                term: {"occ": 1, "ratio": ratio}
                for term, ratio in term_acc["per_term_ratios"].items()
            }
            for term, ratio_info in _preview_term_stats(preview_data, limit=TOP_TERM_PREVIEW):
                print(f"[{mode}]   - {term}: ratio={_fmt_metric(ratio_info['ratio'])}")
        else:
            print(f"[{mode}] Top terminology accuracy terms: N/A")

        per_term_consistency = term_cons.get("per_term") or {}
        if per_term_consistency:
            print(f"[{mode}] Top terminology consistency terms:")
            for term, stats in _preview_term_stats(per_term_consistency, limit=TOP_TERM_PREVIEW):
                print(
                    f"[{mode}]   - {term}: occ={stats.get('occ', 0)}, "
                    f"pseudo_ref={stats.get('pseudo_ref')}, "
                    f"consistency={_fmt_metric(stats.get('consistency'))}"
                )
        else:
            print(f"[{mode}] Top terminology consistency terms: N/A")
    else:
        print(f"[{mode}] Skipping BLEU/chrF — reference field '{REF_FIELD}' not found.")

    print(f"[{mode}] Sample previews:")
    for idx, (sample, pred) in enumerate(
        list(zip(samples, preds))[:TOP_SAMPLE_PREVIEW],
        start=1,
    ):
        ref = sample.get(REF_FIELD)
        print(f"[{mode}]   [{idx}] EN: {sample.get('en', '').strip()}")
        print(f"[{mode}]       PRED: {pred.strip()}")
        if isinstance(ref, str):
            print(f"[{mode}]       REF : {ref.strip()}")
        else:
            print(f"[{mode}]       REF : {ref}")

    print(f"--- Mode: {mode} (finished) ---")
    return {
        "predictions_file": str(output_jsonl),
        "metrics": metrics,
    }


base_output_dir = Path(OUTPUT_DIR)
base_output_dir.mkdir(parents=True, exist_ok=True)

print(f"\n=== File: {DATA_FILE}")
print("=== Translation direction: English → German")
print(f"=== Samples: {len(samples)}")
print(f"=== MAX_SAMPLES: {MAX_SAMPLES}")
print(f"=== Model: {MODEL_NAME}")
print(f"=== Modes (sequential): {MODES}")

all_results = {}

for mode in MODES:
    all_results[mode] = run_mode_evaluation(mode, samples, base_output_dir)

metrics_path = base_output_dir / f"{DATA_STEM}_metrics_summary.json"

with open(str(metrics_path), "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("\nEvaluation finished.")
print("Saved metrics summary to:", metrics_path)
